In [5]:
!pip install datasets pandas matplotlib seaborn spacy
!python -m spacy download en_core_web_sm

Traceback (most recent call last):
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 791, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 492, in _make_request
    raise new_e
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 468, in _make_request
    self._validate_conn(conn)
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 1097, in _validate_conn
    conn.connect()
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connection.py", line 642, in connect
    sock_and_verified = _ssl_wrap_socket_and_match_hostname(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\pavan\anaconda3\Lib\site-packages\urllib3\connection.py", line 783, in _ssl_wrap_socket_and_match_hostname
    ssl_sock = ssl_wrap_socket(
               ^^^^^^^^^^^^^^^^
  Fi

In [6]:
from datasets import load_dataset
import pandas as pd
import re
import spacy
import json

In [7]:
path = "FinEntity.json"
df = pd.read_json(path)
print(df.head())
print(df.columns)

                                             content  \
0  Johnson & Johnson <JNJ.N> shares gained 0.20% ...   
1  On the positive side, Siemens is rallying 6% a...   
2  Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
3  Nearly all major S&P 500 sectors are red, with...   
4  NEW YORK - Wall Street ended sharply higher on...   

                                         annotations  
0  [{'end': 17, 'tag': 'Positive', 'value': 'John...  
1  [{'end': 107, 'tag': 'Positive', 'value': 'Huh...  
2  [{'end': 11, 'tag': 'Positive', 'value': 'Bren...  
3  [{'end': 162, 'tag': 'Positive', 'value': 'hea...  
4  [{'end': 69, 'tag': 'Positive', 'value': 'Tesl...  
Index(['content', 'annotations'], dtype='object')


In [8]:
data = []
for _, row in df.iterrows():
    sentence = row['content']
    for ann in row['annotations']:
        entity = ann.get('value')
        sentiment = ann.get('tag')
        data.append([sentence, entity, sentiment])

df_flat = pd.DataFrame(data, columns=['sentence', 'entity', 'sentiment'])
print("\nFlattened dataset sample:\n", df_flat.head(20))


Flattened dataset sample:
                                              sentence  \
0   Johnson & Johnson <JNJ.N> shares gained 0.20% ...   
1   On the positive side, Siemens is rallying 6% a...   
2   On the positive side, Siemens is rallying 6% a...   
3   Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
4   Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
5   Nearly all major S&P 500 sectors are red, with...   
6   Nearly all major S&P 500 sectors are red, with...   
7   Nearly all major S&P 500 sectors are red, with...   
8   Nearly all major S&P 500 sectors are red, with...   
9   Nearly all major S&P 500 sectors are red, with...   
10  NEW YORK - Wall Street ended sharply higher on...   
11  NEW YORK - Wall Street ended sharply higher on...   
12  Levi Strauss & Co <LEVI.N> gained 2% after its...   
13  However, top oilfield services company Schlumb...   
14      JPMorgan Chase & Co <JPM.N>, Morgan Stanle...   
15      JPMorgan Chase & Co <JPM.N>, Morgan Stanle...   
16 

In [9]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[A-Z]+\.[A-Z]+>', '', text)
    text = re.sub(r'[^A-Za-z0-9.,!?;\'\"\s]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nBefore cleaning:\n", df_flat['sentence'].head(5))
df_flat['sentence'] = df_flat['sentence'].apply(clean_text)
print("\nAfter cleaning:\n", df_flat['sentence'].head(5))


Before cleaning:
 0    Johnson & Johnson <JNJ.N> shares gained 0.20% ...
1    On the positive side, Siemens is rallying 6% a...
2    On the positive side, Siemens is rallying 6% a...
3    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
4    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
Name: sentence, dtype: object

After cleaning:
 0    Johnson Johnson shares gained 0.20 after posti...
1    On the positive side, Siemens is rallying 6 af...
2    On the positive side, Siemens is rallying 6 af...
3    Brent crude rose 1.4 to 100.69 per barrel and ...
4    Brent crude rose 1.4 to 100.69 per barrel and ...
Name: sentence, dtype: object


In [10]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

df_flat['sentence'] = df_flat['sentence'].str.lower().apply(lemmatize_text)

In [11]:
before_drop = len(df_flat)
df_flat.dropna(subset=['sentence'], inplace=True)
df_flat = df_flat[df_flat['sentence'].str.strip() != ""]
after_drop = len(df_flat)
print(f"\nDropped {before_drop - after_drop} empty rows. Remaining rows: {after_drop}")


Dropped 0 empty rows. Remaining rows: 2131


In [12]:
ENTITY_TYPES = [
    "COMPANY",
    "ORGANIZATION",
    "CURRENCY",
    "PERSON",
    "OTHER"
]

In [13]:
def get_entity_type(entity):
    entity_lower = entity.lower()

    if "inc" in entity_lower or "ltd" in entity_lower or "corp" in entity_lower:
        return "COMPANY"
    elif any(keyword in entity_lower for keyword in ["organization", "association", "committee", "foundation", "agency"]):
        return "ORGANIZATION"
    elif entity_lower in ["usd", "eur", "gbp", "inr"]:
        return "CURRENCY"
    elif all(word.isalpha() and word.istitle() for word in entity.split()):
        return "PERSON"
    else:
        return "OTHER"

df_flat['entity_type'] = df_flat['entity'].apply(lambda x: get_entity_type(str(x)))

In [14]:
def get_entity_type_info(entity):
    entity_type = get_entity_type(entity)                       
    return entity_type

entity = "World Health Organization"
etype = get_entity_type_info(entity)
print(f"Entity: {entity}")
print(f"Entity Type: {etype}")

Entity: World Health Organization
Entity Type: ORGANIZATION


In [131]:
SENTIMENTS = ["positive", "negative", "neutral"]
BILU = ["B", "I", "L","U"]

all_labels = ["O"]
for ent_type in ENTITY_TYPES:
    for sent in SENTIMENTS:
        for bilu in BILU:
            all_labels.append(f"{bilu}-{ent_type}-{sent}")

label2id = {label: idx for idx, label in enumerate(all_labels)}
id2label = {idx: label for label, idx in label2id.items()}
entity_type2id = {etype: idx for idx, etype in enumerate(ENTITY_TYPES)}
id2entity_type = {idx: etype for etype, idx in entity_type2id.items()}
mappings = {
    "label2id": label2id,                     # entity-sentiment combined label
    "id2label": id2label,
    "entity_type2id": entity_type2id,         # only entity types
    "id2entity_type": id2entity_type
}

print("\nSample label2id mapping (first 10):", dict(list(label2id.items())[:10]))
print(f"Total Labels: {len(all_labels)}")


Sample label2id mapping (first 10): {'O': 0, 'B-COMPANY-positive': 1, 'I-COMPANY-positive': 2, 'L-COMPANY-positive': 3, 'U-COMPANY-positive': 4, 'B-COMPANY-negative': 5, 'I-COMPANY-negative': 6, 'L-COMPANY-negative': 7, 'U-COMPANY-negative': 8, 'B-COMPANY-neutral': 9}
Total Labels: 61


In [120]:
output_path = "Preprocessed_FinEntity.csv"
df_flat.to_csv(output_path, index=False)
print(f"\nPreprocessing complete! Saved to {output_path}")

with open("label_mappings.json", "w") as f:
    json.dump(mappings, f, indent=4)
print("\nLabel schema & mappings saved to label_mappings.json")


Preprocessing complete! Saved to Preprocessed_FinEntity.csv

Label schema & mappings saved to label_mappings.json


In [123]:
from transformers import AutoTokenizer
finbert_model_name = "ProsusAI/finbert"
FinBERT_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

In [124]:
def get_label2id(entity_type_with_label):
    if(entity_type_with_label == "OTHER"):
                return label2id["OTHER"]
    else:
        return label2id["U"+"-" +entity_type_with_label]

In [169]:
def tokenize_and_align_labels(row):
    
    # Extract content from row
    content = str(row['content'].iloc[0])
    print(f"Sentence:\n{content}\n")
    
    # Tokenize the content
    tokenized_output = FinBERT_tokenizer(content, return_offsets_mapping = True, padding = True, truncation=True)

    # Initialize labels with id of "O" expect begenning and ending of sentence
    # labels values will change based on entities
    labels = [label2id["O"]] * len(tokenized_output.input_ids)
    labels[0] = -100
    labels[-1] =-100

    # input IDs and attention Mask
    input_ids = tokenized_output.input_ids
    attention_mask = tokenized_output.attention_mask
    # Get tokens and offsets from the tokenizer's output
    tokens = FinBERT_tokenizer.convert_ids_to_tokens(tokenized_output["input_ids"])
    offsets = tokenized_output["offset_mapping"]
    print(f"tokens:\n{tokens}\n")

    
    # Loop through each annotation in the row
    for annotation in row['annotations'].iloc[0]:
        # Extract the entity value and get its type from the lookup function
        entity_value = annotation['value']
        entity_type = get_entity_type(entity_value)
        sentiment = annotation['tag'].lower()
        
        
        # Get the character start and end from the annotation
        offset_mappings_from_annotation = [annotation['start'] , annotation['end']]
        
        # Create a list to store tokens that belong to the current entity
        labels_to_change = []
        # Loop through the tokenizer's offsets to find matching tokens
        for offset in range(len(offsets)):
            # Check if the token's character range is within the annotation's range
            if(offsets[offset][0] >= offset_mappings_from_annotation[0] and offsets[offset][1] <= offset_mappings_from_annotation[1]):
                # If there's a match, get the token text from the content
                token_text = content[offsets[offset][0]:offsets[offset][1]]
                # Add the token text to the list if it's not an empty string
                if(token_text not in [""]):
                    labels_to_change.append(offset)

        # nothing to change 
        if not labels_to_change:
            continue
                    
        if(len(labels_to_change)==1):
            # single token entity so U tag
            labels[labels_to_change[0]] = label2id[f"U-{entity_type}-{sentiment}"]
        else:
            # multi - token enities so use B,I,L tags
            labels[labels_to_change[0]] = label2id[f"B-{entity_type}-{sentiment}"]
            
            for i in range(1, len(labels_to_change)-1):
                labels[labels_to_change[i]] = label2id[f"I-{entity_type}-{sentiment}"]
            
            labels[labels_to_change[len(labels_to_change)-1]] = label2id[f"L-{entity_type}-{sentiment}"]
        
    return {"input_ids":input_ids, "attention_mask":attention_mask, "labels": labels}

# now you need to get BILOU taggings of enitities and recreate the sentence lets say and then 

In [171]:
print("SAMPLE OUTPUTS")
test_data = df[:5]
for i in range(len(test_data)):
    print("*"*10)
    output = tokenize_and_align_labels(test_data[i:i+1])
    for k in output:
        print(f"{k}:\n{output[k]}")
        print()
    print("*"*10)

SAMPLE OUTPUTS
**********
Sentence:
Johnson & Johnson <JNJ.N> shares gained 0.20% after posting results that beat expectations but cut its full-year outlook, citing a stronger dollar. [nL4N2Z028U]

tokens:
['[CLS]', 'johnson', '&', 'johnson', '<', 'j', '##n', '##j', '.', 'n', '>', 'shares', 'gained', '0', '.', '20', '%', 'after', 'posting', 'results', 'that', 'beat', 'expectations', 'but', 'cut', 'its', 'full', '-', 'year', 'outlook', ',', 'citing', 'a', 'stronger', 'dollar', '.', '[', 'nl', '##4', '##n', '##2', '##z', '##0', '##28', '##u', ']', '[SEP]']

input_ids:
[101, 3779, 1004, 3779, 1026, 1046, 2078, 3501, 1012, 1050, 1028, 6661, 4227, 1014, 1012, 2322, 1003, 2044, 14739, 3463, 2008, 3786, 10908, 2021, 3013, 2049, 2440, 1011, 2095, 17680, 1010, 8951, 1037, 6428, 7922, 1012, 1031, 17953, 2549, 2078, 2475, 2480, 2692, 22407, 2226, 1033, 102]

attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 